In [69]:
"""
Notebook to pre processing extracted and prepared tax rulings

"""
#import section
import json
import re
import glob
import os
from collections import defaultdict, Counter
import unicodedata
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

In [70]:
#Spanish tokenization models
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [71]:
ruling_tokenized_dict = defaultdict(dict)
list_of_keys = ["accept", "partial", "denied"]
#set containing each word in vocabulary
vocabulary = set()
#counts of words just to check since it shoud be generated only with the test, to avoid data leakage
vocabulary_counts = Counter()

In [72]:
input_dir = "/home/jovyan/prepared_extracted/"
input_file = os.path.join(input_dir, "extracted_classified_tr.json")
with open(input_file, "r", encoding="utf-8") as f:
        json_data = json.load(f)

In [82]:
# To showcase knowledge on vectorization ill use the basic traditional nltk.
# There are better approaches for tokenization. Nevertheless, this approach provides flexibility. 
vocabulary = set()
#create a set of stopwords for spanish
stop_words = set(stopwords.words("spanish"))

#regex pattern to substitute Articles + Art number with a concatenated version since Arts are relevant
#later I´ll ponder wether doing the same for invoked rulings is suitable. In Ecuador, as a rule, rulings are not
#a legal norm. Therefore, the most important source or legal regulation is Arts. of law
art_pattern = r"\b(art(?:[íi]culo)?)\.?\s*(\d+)\b"

for type in list_of_keys:
    ruling_dict = json_data.get(type, {})  # Added {} as default to prevent errors if "root" is missing
    
    # Process each key and its stored values
    for judicial_process, ruling_context in ruling_dict.items():
        #convert arts into one word to avoid loosing important info. 
        #arts are relevant to the sense in which the ruling was issued.
        #Due to their importance theyll be treated as  specific word. 
        art_ruling_context = re.sub(art_pattern, r"\1\2", ruling_context, flags=re.IGNORECASE) 
        ruling_sentences = sent_tokenize(art_ruling_context, language= "spanish")
        sentence_words = []
        for sentence in ruling_sentences:
            words_ruling = word_tokenize(sentence, language='spanish')
            #in order to consider ART17 for example, it is necesarry to consider all alphanumeric but exclude digits
            #excluding digits will eliminate values as they shouldnt have any incidence in the ruling's type
            cleaned_words = [w.lower() for w in words_ruling if w.isalnum() and not w.isdigit() and w not in stop_words]

            if cleaned_words:
                sentence_words.append(cleaned_words)
                vocabulary.update(cleaned_words)
                vocabulary_counts.update(cleaned_words)

        ruling_tokenized_dict[type][judicial_process] = sentence_words
            


In [83]:
print(ruling_tokenized_dict["partial"]["17510201900371"][:20])
print(len(ruling_tokenized_dict["partial"]["17510201900371"]))

[['alegación', 'del', 'vicio', 'por', 'falta', 'de', 'motivación', 'de', 'la', 'motivación', 'del', 'acto', 'administrativo', 'la', 'motivación', 'acto', 'autoridad', 'pública', 'expresión', 'oral', 'escrita', 'razonamiento', 'autoridad', 'busca', 'justificar', 'dicho', 'acto'], ['corte', 'constitucional', 'ecuador', 'sentencia', 'nro'], ['en', 'términos', 'sencillos', 'motivación', 'acto', 'administrativo', 'justificación', 'decisión', 'administración', 'n', 'estado', 'constitucional', 'legitimidad', 'decisiones', 'estatales', 'depende', 'solo', 'quién', 'toma', 'sino', 'porqué', 'hace', 'órgano', 'poder', 'público', 'solo', 'deber', 'ceñir', 'actos', 'competencias', 'procedimientos', 'jurídicamente', 'establecidos', 'legitimidad', 'formal', 'sino', 'deber', 'motivar', 'dichos', 'actos', 'decir', 'fundamentarlos', 'racionalmente', 'legitimidad', 'material'], ['corte', 'constitucional', 'ecuador', 'sentencia', 'nro'], ['el', 'artículo4', 'reglamento', 'control', 'discrecionalidad', 'ac

In [66]:
print(list(vocabulary_counts.items())[:5])

[('sponde', 2), ('analizar', 702), ('validez', 574), ('legalidad', 1657), ('acto', 8912)]


In [67]:
print(list(vocabulary)[:5])

['consignatario', 'desarrollando', 'encuentre', 'actualmente', 'tributan']
